In [2]:
from sentence_transformers import SentenceTransformer
from transformers import T5ForConditionalGeneration, T5Tokenizer
from langchain.text_splitter import RecursiveCharacterTextSplitter
import faiss
import numpy as np
import torch
import os

In [9]:
raw_documents = [
    "India is a country in South Asia. It has 28 states and 8 Union Territories. The capital is New Delhi. Hindi is the official language.",
    "The Indian economy is one of the largest in the world. It is driven by agriculture, manufacturing, and services.",
    "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. It is named after Gustave Eiffel."
]

splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=10)
documents = []
for doc in raw_documents:
    print(doc)
    chunks = splitter.split_text(doc)
    print("chunk:", chunks)
    documents.extend(chunks)

print('\n'.join(documents))


India is a country in South Asia. It has 28 states and 8 Union Territories. The capital is New Delhi. Hindi is the official language.
chunk: ['India is a country in South Asia. It has 28 states', '28 states and 8 Union Territories. The capital is', 'is New Delhi. Hindi is the official language.']
The Indian economy is one of the largest in the world. It is driven by agriculture, manufacturing, and services.
chunk: ['The Indian economy is one of the largest in the', 'in the world. It is driven by agriculture,', 'manufacturing, and services.']
The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. It is named after Gustave Eiffel.
chunk: ['The Eiffel Tower is a wrought-iron lattice tower', 'tower on the Champ de Mars in Paris, France. It', 'It is named after Gustave Eiffel.']
India is a country in South Asia. It has 28 states
28 states and 8 Union Territories. The capital is
is New Delhi. Hindi is the official language.
The Indian economy is one of the la

In [10]:
embedder = SentenceTransformer('all-mpnet-base-v2')

c:\Users\91936\Desktop\RAG\env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\91936\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to r

In [11]:
doc_embeddings = embedder.encode(documents, convert_to_tensor=False)

In [13]:
doc_embeddings.shape  

(9, 768)

In [17]:
dim = doc_embeddings[0].shape[0]
print(dim)
index = faiss.IndexFlatL2(dim)
index.add(np.array(doc_embeddings))

768


In [19]:
def retrieve(query, k=3):
    query_embedding = embedder.encode([query], convert_to_tensor=False)
    distances, indices = index.search(np.array(query_embedding), k)
    print("Distances:", distances)
    print("Indices:", indices)
    return [documents[i] for i in indices[0]]

In [20]:
tokenizer = T5Tokenizer.from_pretrained("t5-base")
model = T5ForConditionalGeneration.from_pretrained("t5-base")

def generate_answer(query, retrieved_docs):
    context = " ".join(retrieved_docs)
    input_text = f"question: {query} context: {context}"
    inputs = tokenizer.encode(input_text, return_tensors="pt", truncation=True, max_length=512)
    outputs = model.generate(inputs, max_length=100)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

c:\Users\91936\Desktop\RAG\env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\91936\.cache\huggingface\hub\models--t5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, a

In [21]:
query = "What is the capital of India?"
retrieved_docs = retrieve(query, k=3)   
answer = generate_answer(query, retrieved_docs)
print("Query:", query)
print("Retrieved Documents:", retrieved_docs)
print("Generated Answer:", answer)

Distances: [[0.8023462  0.80871344 1.0274394 ]]
Indices: [[2 0 3]]
Query: What is the capital of India?
Retrieved Documents: ['is New Delhi. Hindi is the official language.', 'India is a country in South Asia. It has 28 states', 'The Indian economy is one of the largest in the']
Generated Answer: New Delhi


In [22]:
query = "What is Indian economy driven by?"
retrieved_docs = retrieve(query, k=3)
answer = generate_answer(query, retrieved_docs)
print("Query:", query)
print("Retrieved Documents:", retrieved_docs)
print("Generated Answer:", answer)

Distances: [[0.4856503 0.7772261 1.0823958]]
Indices: [[3 4 5]]
Query: What is Indian economy driven by?
Retrieved Documents: ['The Indian economy is one of the largest in the', 'in the world. It is driven by agriculture,', 'manufacturing, and services.']
Generated Answer: agriculture, manufacturing, and services
